In [ ]:
#  DATA GENERATOR
# ============================================================

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

# -----------------------------
# CONFIG
# -----------------------------
np.random.seed(42)
random.seed(42)

N_CUSTOMERS = 20_000
N_CATEGORIES = 50
N_SUPPLIERS = 500
N_PRODUCTS = 1_000
N_ORDERS = 100_000

# ============================================================
# 1. CATEGORIES
# ============================================================

category_names = [
    "Electronics", "Fashion", "Beauty", "Home & Living",
    "Sports", "Automotive", "Books", "Food & Beverage",
    "Health", "Toys"
]

# Tambahkan sampai 50 kategori
category_names += [
    f"Category {i}" for i in range(len(category_names) + 1, N_CATEGORIES + 1)
]

categories = pd.DataFrame({
    "category_id": range(1, N_CATEGORIES + 1),
    "category_name": category_names
})

# ============================================================
# 2. SUPPLIERS
# ============================================================

cities = [
    "Jakarta", "Bandung", "Bekasi", "Depok", "Tangerang",
    "Bogor", "Semarang", "Yogyakarta", "Surabaya",
    "Malang", "Solo", "Medan", "Makassar", "Palembang",
    "Denpasar"
]

suppliers = pd.DataFrame({
    "supplier_id": range(1, N_SUPPLIERS + 1),
    "supplier_name": [f"Supplier {i}" for i in range(1, N_SUPPLIERS + 1)],
    "city": np.random.choice(cities, N_SUPPLIERS),
    "province": np.random.choice(
        ["DKI Jakarta", "Jawa Barat", "Jawa Tengah",
         "DI Yogyakarta", "Jawa Timur", "Sumatera Utara",
         "Sulawesi Selatan", "Bali"],
        N_SUPPLIERS
    )
})

# ============================================================
# 3. PRODUCTS
# ============================================================

products = pd.DataFrame({
    "product_id": range(1, N_PRODUCTS + 1),
    "product_name": [f"Product {i}" for i in range(1, N_PRODUCTS + 1)],
    "category_id": np.random.randint(1, N_CATEGORIES + 1, N_PRODUCTS),
    "supplier_id": np.random.randint(1, N_SUPPLIERS + 1, N_PRODUCTS),
    "brand": [f"Brand {random.randint(1, 100)}" for _ in range(N_PRODUCTS)],
})

# Harga modal
products["cost_price"] = np.random.randint(
    20_000, 5_000_000, N_PRODUCTS
)

# Harga jual = modal + margin 10%-80%
margin = np.random.uniform(1.10, 1.80, N_PRODUCTS)

products["selling_price"] = (
    products["cost_price"] * margin
).round(-3).astype(int)

# Stock
products["stock"] = np.random.randint(0, 1_000, N_PRODUCTS)

# ============================================================
# 4. CUSTOMERS
# ============================================================

start_registration = datetime(2023, 1, 1)

registration_days = np.random.randint(
    0, 365 * 3, N_CUSTOMERS
)

customers = pd.DataFrame({
    "customer_id": range(1, N_CUSTOMERS + 1),
    "customer_name": [
        f"Customer {i}" for i in range(1, N_CUSTOMERS + 1)
    ],
    "gender": np.random.choice(
        ["Male", "Female"], N_CUSTOMERS
    ),
    "birth_date": pd.to_datetime(
        np.random.randint(
            int(pd.Timestamp("1970-01-01").timestamp()),
            int(pd.Timestamp("2005-12-31").timestamp()),
            N_CUSTOMERS
        ),
        unit="s"
    ),
    "city": np.random.choice(cities, N_CUSTOMERS),
    "province": np.random.choice(
        ["DKI Jakarta", "Jawa Barat", "Jawa Tengah",
         "DI Yogyakarta", "Jawa Timur", "Sumatera Utara",
         "Sulawesi Selatan", "Bali"],
        N_CUSTOMERS
    ),
    "customer_segment": np.random.choice(
        ["Regular", "Silver", "Gold", "Platinum"],
        N_CUSTOMERS,
        p=[0.55, 0.25, 0.15, 0.05]
    ),
    "registration_date": [
        start_registration + timedelta(days=int(x))
        for x in registration_days
    ]
})

# ============================================================
# 5. ORDERS
# ============================================================

# Customer tertentu dibuat lebih aktif
customer_weights = np.random.exponential(
    scale=1,
    size=N_CUSTOMERS
)
customer_weights = customer_weights / customer_weights.sum()

order_customer = np.random.choice(
    customers["customer_id"],
    N_ORDERS,
    p=customer_weights
)

order_start = datetime(2024, 1, 1)

order_dates = [
    order_start + timedelta(
        days=random.randint(0, 730),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    for _ in range(N_ORDERS)
]

orders = pd.DataFrame({
    "order_id": range(1, N_ORDERS + 1),
    "customer_id": order_customer,
    "order_date": order_dates,
    "order_status": np.random.choice(
        ["Completed", "Cancelled", "Returned"],
        N_ORDERS,
        p=[0.90, 0.06, 0.04]
    )
})

# ============================================================
# 6. ORDER ITEMS
# ============================================================

# Rata-rata 3 item per order
items_per_order = np.random.choice(
    [1, 2, 3, 4, 5],
    N_ORDERS,
    p=[0.15, 0.30, 0.30, 0.18, 0.07]
)

order_item_records = []
item_id = 1

for order_id, item_count in zip(
    orders["order_id"],
    items_per_order
):
    selected_products = np.random.choice(
        products["product_id"],
        item_count,
        replace=False
    )

    for product_id in selected_products:

        product_price = products.loc[
            products["product_id"] == product_id,
            "selling_price"
        ].iloc[0]

        quantity = np.random.randint(1, 6)

        discount = np.random.choice(
            [0, 0.05, 0.10, 0.15, 0.20],
            p=[0.50, 0.20, 0.15, 0.10, 0.05]
        )

        order_item_records.append([
            item_id,
            order_id,
            product_id,
            quantity,
            product_price,
            discount
        ])

        item_id += 1

order_items = pd.DataFrame(
    order_item_records,
    columns=[
        "order_item_id",
        "order_id",
        "product_id",
        "quantity",
        "unit_price",
        "discount"
    ]
)

# ============================================================
# 7. PAYMENTS
# ============================================================

payments = pd.DataFrame({
    "payment_id": range(1, N_ORDERS + 1),
    "order_id": orders["order_id"],
    "payment_date": orders["order_date"] + pd.to_timedelta(
        np.random.randint(0, 2, N_ORDERS),
        unit="D"
    ),
    "payment_method": np.random.choice(
        ["Bank Transfer", "E-Wallet", "Credit Card", "COD"],
        N_ORDERS,
        p=[0.30, 0.35, 0.20, 0.15]
    )
})

# Payment status mengikuti order
payments["payment_status"] = np.where(
    orders["order_status"] == "Cancelled",
    "Failed",
    np.random.choice(
        ["Paid", "Pending"],
        N_ORDERS,
        p=[0.95, 0.05]
    )
)

# ============================================================
# 8. SHIPMENTS
# ============================================================

shipping_date = orders["order_date"] + pd.to_timedelta(
    np.random.randint(1, 4, N_ORDERS),
    unit="D"
)

delivery_date = shipping_date + pd.to_timedelta(
    np.random.randint(1, 8, N_ORDERS),
    unit="D"
)

shipments = pd.DataFrame({
    "shipment_id": range(1, N_ORDERS + 1),
    "order_id": orders["order_id"],
    "shipping_date": shipping_date,
    "delivery_date": delivery_date,
    "shipping_method": np.random.choice(
        ["Regular", "Express", "Same Day"],
        N_ORDERS,
        p=[0.65, 0.25, 0.10]
    ),
    "shipping_cost": np.random.randint(
        10_000, 100_000, N_ORDERS
    ),
    "shipment_status": np.random.choice(
        ["Delivered", "In Transit", "Returned"],
        N_ORDERS,
        p=[0.88, 0.08, 0.04]
    )
})

# ============================================================
# 9. SAVE CSV
# ============================================================

tables = {
    "customers": customers,
    "categories": categories,
    "suppliers": suppliers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "payments": payments,
    "shipments": shipments
}

for name, df in tables.items():
    df.to_csv(f"{name}.csv", index=False)

# ============================================================
# 10. CHECK DATASET
# ============================================================

print("=== DATASET BERHASIL DIBUAT ===\n")

for name, df in tables.items():
    print(f"{name:15} : {len(df):,} rows")

print("\n=== SAMPLE ORDERS ===")
display(orders.head())

print("\n=== SAMPLE ORDER ITEMS ===")
display(order_items.head())